In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd

from pathlib import Path

In [ ]:
%%RecordEvent
%%time
### cell 0 ###

benchmark_name = "what-course-are-you-going-to-take"
course = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "course-reviews-university-of-waterloo" / "course_data_clean.csv"
)
factor = 300
course = pd.concat([course] * factor, ignore_index=True)
course.info()

In [ ]:
%%RecordEvent
%%time
### cell 1 ###

course.head(10)

In [ ]:
%%RecordEvent
%%time
### cell 2 ###

course.tail(10)

In [ ]:
%%RecordEvent
%%time
### cell 3 ###

course.describe()

In [ ]:
%%RecordEvent
%%time
### cell 4 ###

course.info()

In [ ]:
%%RecordEvent
%%time
### cell 5 ###

course = course.dropna()

In [ ]:
%%RecordEvent
%%time
### cell 6 ###

course[["course_unit", "course_num"]] = course["course_code"].str.split(
    " ", expand=True
)

In [ ]:
%%RecordEvent
%%time
### cell 7 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 8 ###

course[course["num_reviews"] < 10].index

In [ ]:
%%RecordEvent
%%time
### cell 9 ###


def extract_filter_features(df, col_name, threshold):
    """Extracts features for predicting execution time of filtering operation."""

    num_rows = len(df)
    dtype_str = str(df[col_name].dtype)  # Get dtype as string
    nan_ratio = df[col_name].isna().mean()  # Compute NaN ratio

    # Compute selectivity: fraction of rows passing the filter
    if df[col_name].dropna().empty:
        target_selectivity = 0.0
    else:
        target_selectivity = (df[col_name] < threshold).mean()

    return {
        "num_rows": num_rows,
        "dtype_str": dtype_str,
        "nan_ratio": nan_ratio,
        "target_selectivity": target_selectivity,
    }


extract_filter_features(course, "num_reviews", 10)

In [ ]:
%%RecordEvent
%%time
### cell 10 ###

course.drop(course[course["num_reviews"] < 10].index, inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 11 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 12 ###

for i_1 in ["useful", "easy", "liked"]:
    course[i_1] = course[i_1].str.replace("%", "")
    course[i_1] = course[i_1].astype("int")

In [ ]:
%%RecordEvent
%%time
### cell 13 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 14 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 15 ###

course.drop(["course_title", "reviews", "course_rating"], axis=1, inplace=True)

In [ ]:
%%RecordEvent
%%time
### cell 16 ###

course

In [ ]:
%%RecordEvent
%%time
### cell 17 ###

course.info()

In [ ]:
%%RecordEvent
%%time
### cell 18 ###

course_gp = course.groupby("course_unit").mean(numeric_only=True)

In [ ]:
%%RecordEvent
%%time
### cell 19 ###

course_gp.head()

In [ ]:
%%RecordEvent
%%time
### cell 20 ###

course["course_rating_mean"] = None
for i_2 in course_gp.index:
    course.loc[i_2, "course_rating_mean"] = course_gp.loc[i_2, "course_rating_int"]

In [ ]:
### cell 21 ###

course

In [ ]:
### cell 22 ###

# Use the GPU‐native reset_index (inplace isn’t supported, so assign the result back)
course = course.reset_index()

In [ ]:
### cell 23 ###

course

In [ ]:
### cell 24 ###

course.groupby("course_unit")["course_rating_int"].mean()

In [ ]:
### cell 25 ###

def extract_features_from_dataframe(df, group_column, agg_function="mean"):    
    """Extracts features for predicting the execution time of a groupby operation."""
    # total rows and columns (metadata on the GPU is just Python ints)
    n_rows = int(df.shape[0])
    n_cols = int(df.shape[1])

    # compute number of groups exactly as in the original code
    n_groups = int(df[group_column].nunique())
    # compute group sizes once for the max
    group_sizes = df[group_column].value_counts()
    max_group_size = int(group_sizes.max())

    # count column dtypes (CPU metadata access) exactly as in the original
    int_count = int(df.select_dtypes(include=["int", "int32", "int64"]).shape[1])
    float_count = int(df.select_dtypes(include=["float", "float32", "float64"]).shape[1])
    str_count = int(df.select_dtypes(include=["object", "string"]).shape[1])

    # encode aggregation function
    agg_map = {"sum": 0, "mean": 1, "count": 2}
    agg_function_encoded = agg_map.get(agg_function, 1)

    return {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "group_range": n_groups,  # alias for n_groups
        "n_groups": n_groups,
        "max_group_size": max_group_size,
        "int_count": int_count,
        "float_count": float_count,
        "str_count": str_count,
        "agg_function": agg_function_encoded,
    }

# call the function so the cell returns the features dict
extract_features_from_dataframe(course, "course_unit", "mean")

In [ ]:
### cell 26 ###

course[course["course_code"].str.startswith("CS")].value_counts()

In [ ]:
### cell 27 ###

course

In [ ]:
### cell 28 ###

# Fall back to the in-place sort to guarantee bit-for-bit equivalence with the original output
course.sort_values("course_rating_mean", ascending=False, inplace=True)

In [ ]:
### cell 29 ###

course = course.reset_index()

In [ ]:
### cell 30 ###

course.set_index("course_unit", inplace=True)

In [ ]:
### cell 31 ###

course['course_rating_mean'][course.index == 'KOREA'].value_counts()

In [ ]:
### cell 32 ###

KOREA = course[course.index == "KOREA"]

In [ ]:
### cell 33 ###

course.loc[course.index == "CHINA", "course_rating_mean"].value_counts()

In [ ]:
### cell 34 ###
# build a GPU‐side boolean mask instead of using get_loc + iloc
mask = course.index == 'CHINA'
# boolean‐mask indexing via .loc is a fully vectorized GPU operation
china = course.loc[mask, :]

In [ ]:
### cell 35 ###

course.loc[course.index == "CHINA", "course_rating_mean"].value_counts()

In [ ]:
### cell 36 ###

span = course[course.index == "SPAN"].squeeze()

In [ ]:
### cell 37 ###

course.loc[course.index == "CS", "course_rating_mean"].value_counts()

In [ ]:
### cell 38 ###

cs = course[course.index.get_level_values(0) == 'CS']
cs

In [ ]:
### cell 39 ###
# Drop non‐numeric columns before computing the group means (cudf.groupby.mean doesn’t support numeric_only)
# First build a list of numeric columns
numeric_cols = cs.select_dtypes(include=['number']).columns.tolist()
# Ensure we still have the grouping key
cols_to_use = ['course_code'] + numeric_cols
# Compute the mean of only the numeric columns on the GPU
cs_mean = (
    cs[cols_to_use]
      .groupby('course_code')
      .mean()
      .sort_values('course_rating_int', ascending=False)
)
cs_mean

In [ ]:
### cell 40 ###

course['course_rating_mean'][course.index == 'WKRPT'].value_counts()

In [ ]:
### cell 41 ###
wkrpt = course.loc["WKRPT", :]
wkrpt

In [ ]:
### cell 42 ###
# cudf’s GroupBy.mean doesn’t accept numeric_only, so explicitly pick numeric columns before aggregation
numeric_cols = wkrpt.select_dtypes(include=['number']).columns

wkrpt_mean = (
    wkrpt
      .groupby("course_code")[numeric_cols]   # slice to only numeric columns
      .mean()                                   # GPU mean over all numeric cols
      .sort_values("course_rating_int", ascending=False)  # GPU sort by rating
)

wkrpt_mean

In [ ]:
### cell 43 ###

wkrpt.groupby("course_code").mean(numeric_only=True)

In [ ]:
### cell 44 ###

course['course_rating_mean'][course.index == 'PD'].value_counts()

In [ ]:
### cell 45 ###

pd_df = course[course.index == "PD"]
pd_df

In [ ]:
### cell 46 ###

# Pre-select numeric columns to avoid the numeric_only fallback
numeric_cols = pd_df.select_dtypes(include=['number']).columns

pd_mean = (
    pd_df
    .groupby('course_code')[numeric_cols]  # only numeric cols
    .mean()                                # runs on GPU
    .sort_values('course_rating_int', ascending=False)
)

pd_mean